In [109]:
import numpy as np
import pandas as pd
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

price = df[['Price']].values

from sklearn.ensemble import RandomForestRegressor

# ======================
# FEATURE ENGINEERING
# ======================
df_rf = df.copy()
df_rf["Lag1"] = df_rf["Price"].shift(1)
df_rf["Diff"] = df_rf["Price"] - df_rf["Lag1"]
df_rf = df_rf.dropna()

# ======================
# SPLIT
# ======================
split = int(len(df_rf) * 0.8)

train = df_rf.iloc[:split]
test = df_rf.iloc[split:]

X_train = train[["Lag1"]]
y_train = train["Diff"]

X_test = test[["Lag1"]]
y_test = test["Diff"]

actual = test["Price"]

# ======================
# MODEL
# ======================
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

# ======================
# PREDICT
# ======================
pred_diff = rf.predict(X_test)
pred = X_test["Lag1"].values + pred_diff

# ======================
# EVALUATION
# ======================
rmse = np.sqrt(mean_squared_error(actual, pred))
mae = mean_absolute_error(actual, pred)
r2 = r2_score(actual, pred)
mape = np.mean(np.abs((actual - pred) / actual)) * 100

print("\n=== RANDOM FOREST ===")
print(f"MAE: {mae:.6f}") 
print(f"RMSE: {rmse:.6f}") 
print(f"R2: {r2:.6f}") 
print(f"MAPE: {mape:.6f}")



=== RANDOM FOREST ===
MAE: 836.319064
RMSE: 1094.651249
R2: 0.997177
MAPE: 0.985626


In [111]:
os.makedirs("models", exist_ok=True)
joblib.dump(rf, "models/random_forest.pkl")

['models/random_forest.pkl']